In [1]:
from pathlib import Path

import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error


In [6]:

# ============================================================
# CONFIG
# ============================================================

FILE = Path("../data/data/processed/race_laps_2024.parquet")

CIRCUIT = "Barcelona"


# ============================================================
# LOAD DATA
# ============================================================

df = pd.read_parquet(FILE)

df = df[
    (df["Race"] == CIRCUIT) &
    (df["Compound"].isin(["SOFT", "MEDIUM", "HARD"]))
].copy()


# ============================================================
# FEATURES
# ============================================================

features = [
    "LapNumber",
    "TyreLife",
    "Compound",
    "Driver",
    "Team",
]

target = "LapTimeSeconds"

X = df[features]
y = df[target]


# ============================================================
# PREPROCESSING
# ============================================================

categorical_features = [
    "Compound",
    "Driver",
    "Team",
]

numeric_features = [
    "LapNumber",
    "TyreLife",
]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features,
        ),
        (
            "numeric",
            "passthrough",
            numeric_features,
        ),
    ]
)


# ============================================================
# MODEL
# ============================================================

model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", LinearRegression()),
    ]
)


# ============================================================
# TRAIN
# ============================================================

model.fit(X, y)

predictions = model.predict(X)


# ============================================================
# METRICS
# ============================================================

mae = mean_absolute_error(y, predictions)
rmse = mean_squared_error(y, predictions) ** 0.5

print("=" * 70)
print("MODELO B - DEGRADACIÓN")
print("=" * 70)

print(f"Circuito: {CIRCUIT}")
print(f"Filas: {len(df)}")

print()
print("MAE :", round(mae, 3), "s")
print("RMSE:", round(rmse, 3), "s")


# ============================================================
# COEFFICIENTS
# ============================================================

feature_names = model.named_steps[
    "preprocessor"
].get_feature_names_out()

coefficients = model.named_steps[
    "model"
].coef_

coef_df = pd.DataFrame(
    {
        "Feature": feature_names,
        "Coefficient": coefficients,
    }
)

coef_df["AbsCoefficient"] = coef_df["Coefficient"].abs()

coef_df = coef_df.sort_values(
    "AbsCoefficient",
    ascending=False,
)


print()
print("=" * 70)
print("COEFICIENTES")
print("=" * 70)

print(
    coef_df[
        ["Feature", "Coefficient"]
    ].to_string(index=False)
)

MODELO B - DEGRADACIÓN
Circuito: Barcelona
Filas: 1192

MAE : 0.416 s
RMSE: 0.608 s

COEFICIENTES
                          Feature  Coefficient
       categorical__Team_Williams     0.764104
          categorical__Driver_SAR     0.654732
             categorical__Team_RB     0.579966
        categorical__Team_McLaren    -0.543292
       categorical__Team_Mercedes    -0.541490
          categorical__Driver_VER    -0.535812
categorical__Team_Red Bull Racing    -0.442508
          categorical__Driver_NOR    -0.433934
          categorical__Driver_MAG     0.433246
        categorical__Team_Ferrari    -0.432360
          categorical__Driver_BOT     0.389894
          categorical__Driver_TSU     0.386042
    categorical__Team_Kick Sauber     0.286438
          categorical__Driver_HAM    -0.273403
          categorical__Driver_RUS    -0.268086
          categorical__Driver_LEC    -0.250661
          categorical__Driver_HUL    -0.240292
   categorical__Team_Aston Martin     0.234327
         